# Ecuador SNI — interactive stress analytics

A demonstrative analytics notebook using the saved Docker simulation outputs. Each chart is paired with an interpretation: the goal is not only to display results, but to communicate what they mean.

**Scope:** deterministic scenario comparison, sensitivity analysis, time-series aggregation, and model diagnostics. These results are not a probabilistic adequacy study.

In [1]:
# Repository root is discovered; only edit the relative results folder below.
from pathlib import Path
_start = Path.cwd().resolve()
ROOT = next((candidate for candidate in (_start, *_start.parents) if (candidate / '.git').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root from the notebook working directory.')
RESULTS_DIR = ROOT / r'results\docker_runs\phase11-20261027-174819' # or replace for the full resutls dir

In [2]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

root = Path(RESULTS_DIR).expanduser().resolve()
analysis = root / 'analysis'
if not analysis.exists():
    raise FileNotFoundError(f'Expected analysis folder not found: {analysis}')

def load(name):
    return pd.read_csv(analysis / name)

annual = load('phase08a_annual_kpis.csv')
monthly = load('phase08a_monthly_kpis.csv')
duration = load('phase08a_duration_curves.csv')
future = load('phase08_operating_indicators.csv')
carrier = load('phase08_carrier_energy.csv')
print(f'Loaded {root}')
print(f"Hydro scenarios: {', '.join(annual.scenario)}")

Loaded C:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\results\docker_runs\phase11-20261027-174819
Hydro scenarios: H00, H10, H20, H30, H40, H50


## 1. Network topology on the map

A visual starting point: the modeled transmission topology, with every bus shown as a colored point and every modeled line drawn over OpenStreetMap. Hover over a bus or line to inspect its identity.

In [3]:
import pypsa
topology = pypsa.Network(str(root / 'results' / 'phase_06' / 'network_solved_2022.nc'))
bus_view = topology.buses.copy()
bus_view['bus'] = bus_view.index.astype(str)
bus_view['bus_label'] = bus_view['bus']
bus_view['voltage_kv'] = bus_view['v_nom'].round().astype(int)
voltage_levels = sorted(bus_view['voltage_kv'].dropna().unique())
voltage_colors = {level: color for level, color in zip(voltage_levels, ['#1f77b4', '#ff7f0e', '#2ca02c'])}
fig = go.Figure()
for _, row in topology.lines.iterrows():
    b0, b1 = row.bus0, row.bus1
    fig.add_trace(go.Scattermap(lon=[topology.buses.at[b0,'x'], topology.buses.at[b1,'x']], lat=[topology.buses.at[b0,'y'], topology.buses.at[b1,'y']], mode='lines', line={'width':1, 'color':voltage_colors[int(round(row.v_nom))]}, hovertemplate=f"Line {row.name}<br>{b0} → {b1}<extra></extra>"))
for level in voltage_levels:
    bus_group = bus_view[bus_view['voltage_kv'].eq(level)]
    fig.add_trace(go.Scattermap(lon=bus_group.x, lat=bus_group.y, mode='markers+text', text=bus_group.bus_label, textposition='top center', marker={'size':8, 'color':voltage_colors[level]}, name=f'{level} kV', legendgroup=f'{level} kV', showlegend=True, customdata=bus_group[['bus']].to_numpy(), hovertemplate=f'Bus %{{customdata[0]}}<br>Voltage: {level} kV<extra></extra>'))
fig.update_layout(map={'style':'open-street-map','center':{'lat':-1.5,'lon':-78.5},'zoom':5}, title='Ecuador SNI modeled topology — bus-colored network view', template='plotly_white', showlegend=False, height=675)
fig.show()

INFO:pypsa.io:Imported network network_solved_2022.nc has buses, carriers, generators, lines, links, loads, transformers


**Insight:** this map provides the spatial frame for all later results: generation, shedding, and loading are interpreted relative to the same modeled network geometry. It is an illustrative topology view, not a geographic verification of the real grid.

## 1. The main message: stress accelerates after flexibility is exhausted

The curve is intentionally interactive: hover over each point to see the scenario, shedding percentage, and import dependence. The shape communicates that the system absorbs early hydro losses, then becomes increasingly exposed to unmet demand.

In [4]:
fig = px.line(annual, x='hydro_reduction_pct', y='load_shedding_gwh', markers=True,
              color_discrete_sequence=['#d62728'],
              labels={'hydro_reduction_pct':'Selected Sep–Dec hydro-flow reduction (%)', 'load_shedding_gwh':'Annual load shedding (GWh)'},
              title='Load shedding rises nonlinearly as hydro scarcity deepens',
              hover_data={'scenario':True, 'load_shedding_pct':':.3f', 'imports_gwh':':.1f', 'hydro_reduction_pct':False})
fig.update_traces(line_width=4, marker_size=11)
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

**Insight:** imports and thermal generation provide a buffer at low stress, but that buffer is finite. H50 reaches 373.06 GWh of shedding—about 73 times H00’s 5.13 GWh baseline. Because H00 already sheds, later values should be interpreted as incremental stress, not a clean onset threshold.

## 2. Where the energy goes

This stacked view makes the substitution mechanism visible: hydro falls, while oil, other renewables, and imports respond before shedding becomes dominant.

In [5]:
energy = annual.melt(id_vars=['scenario', 'hydro_reduction_pct'],
                  value_vars=['hydro_gwh','thermal_oil_gwh','other_renewables_gwh','imports_gwh','load_shedding_gwh'],
                  var_name='component', value_name='gwh')
energy['component'] = energy.component.map({'hydro_gwh':'Hydro','thermal_oil_gwh':'Oil','other_renewables_gwh':'Other renewables','imports_gwh':'Imports','load_shedding_gwh':'Unserved demand'})
fig = px.area(energy, x='scenario', y='gwh', color='component',
              category_orders={'scenario': list(annual.scenario)},
              labels={'gwh':'Annual energy (GWh)','scenario':'Hydro-stress case','component':'Energy source / outcome'},
              title='The system replaces lost hydro with fuel, imports—and eventually unmet demand',
              color_discrete_map={'Hydro':'#1f77b4','Oil':'#ff7f0e','Other renewables':'#2ca02c','Imports':'#9467bd','Unserved demand':'#d62728'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

**Insight:** the stress response is not simply “less hydro equals less supply.” It is a substitution chain. In H50, oil generation rises by roughly 1.9 TWh relative to H00, imports rise by about 0.3 TWh, and the remaining gap appears as shedding.

## 3. Seasonality: when does the problem happen?

The heatmap separates the imposed September–December shock from the rest of the year. Hover to inspect monthly GWh and shedding hours.

In [6]:
monthly['month_name'] = pd.to_datetime(monthly.month, format='%m').dt.strftime('%b')
month_order = pd.to_datetime(monthly.month, format='%m').dt.strftime('%b').drop_duplicates().tolist()
fig = px.density_heatmap(monthly, x='month_name', y='scenario', z='load_shedding_gwh',
                         category_orders={'month_name':month_order, 'scenario':list(annual.scenario)},
                         color_continuous_scale='Reds',
                         labels={'load_shedding_gwh':'Load shedding (GWh)','month_name':'Month','scenario':'Case'},
                         title='Load shedding is concentrated in the late-year stress season',
                         text_auto='.2f')
fig.update_layout(template='plotly_white')
fig.show()

### Peak hourly demand gap by month

Use the buttons to compare the monthly maximum hourly shedding, imports, and their **coincident** sum in MW. Shedding is demand actually unserved in the model; imports cover demand, so the combined value represents the peak gap that domestic supply alone did not cover. In H50, that combined peak is about 1,638 MW (about 1,188 MW shed plus 450 MW imported).

In [7]:
import pypsa

scenario_order = list(annual.scenario)
peak_import_rows = []
for scenario in scenario_order:
    solved = pypsa.Network(str(root / 'results' / 'phase_08a' / f'{scenario}.solved.nc'))
    import_ids = solved.generators.index[solved.generators.carrier.eq('interconnection_colombia')]
    hourly_imports = solved.generators_t.p[import_ids].sum(axis=1).clip(lower=0)
    for month, peak in hourly_imports.groupby(pd.DatetimeIndex(hourly_imports.index).month).max().items():
        peak_import_rows.append({'scenario': scenario, 'month': month, 'peak_imports_mw': peak})

peak_monthly = monthly.merge(pd.DataFrame(peak_import_rows), on=['scenario', 'month'], validate='one_to_one')
metrics = {
    'Shedding': 'peak_shedding_mw',
    'Imports': 'peak_imports_mw',
    'Shedding + Imports': 'peak_combined_stress_mw',
}
heatmap_values = {label: peak_monthly.pivot(index='scenario', columns='month', values=column)
                  .reindex(index=scenario_order, columns=range(1, 13)).to_numpy()
                  for label, column in metrics.items()}
assert all(pd.notna(values).all() for values in heatmap_values.values())
hover = lambda label: f'Month: %{{x}}<br>Case: %{{y}}<br>Peak {label.lower()}: %{{z:,.1f}} MW<extra></extra>'
fig = go.Figure(go.Heatmap(z=heatmap_values['Shedding + Imports'], x=month_order, y=scenario_order,
                           colorscale='Reds', zmin=0, zmax=peak_monthly.peak_combined_stress_mw.max(),
                           texttemplate='%{z:,.0f}', hovertemplate=hover('Shedding + Imports'),
                           colorbar={'title': 'Peak (MW)'}))
fig.update_layout(template='plotly_white', title='Peak hourly shedding + imports by month and case',
                  xaxis_title='Month', yaxis_title='Case',
                  updatemenus=[{'type': 'buttons', 'direction': 'right', 'x': 0, 'y': 1.16, 'active': 2,
                                'buttons': [{'label': label, 'method': 'update',
                                             'args': [{'z': [values], 'hovertemplate': [hover(label)]},
                                                      {'title': f'Peak hourly {label.lower()} by month and case'}]}
                                            for label, values in heatmap_values.items()]}])
fig.show()

INFO:pypsa.io:Imported network H00.solved.nc has buses, carriers, generators, lines, links, loads, transformers
INFO:pypsa.io:Imported network H10.solved.nc has buses, carriers, generators, lines, links, loads, transformers
INFO:pypsa.io:Imported network H20.solved.nc has buses, carriers, generators, lines, links, loads, transformers
INFO:pypsa.io:Imported network H30.solved.nc has buses, carriers, generators, lines, links, loads, transformers
INFO:pypsa.io:Imported network H40.solved.nc has buses, carriers, generators, lines, links, loads, transformers
INFO:pypsa.io:Imported network H50.solved.nc has buses, carriers, generators, lines, links, loads, transformers


**Insight:** the shortage is temporally concentrated rather than evenly distributed. This matters operationally: monthly or annual averages alone would hide the late-year periods where imports, dispatchable generation, and demand response are most valuable.

## 4. Demand growth: the same portfolio behaves differently

The 2030 cases hold the renewable additions constant and change the demand trajectory. That makes this a controlled comparison of demand pressure.

In [8]:
view = future.copy()
view['case_label'] = view.case.str.replace('_2030','').str.title()
long = view.melt(id_vars='case_label', value_vars=['imports_gwh','load_shedding_gwh'], var_name='metric', value_name='gwh')
long['metric'] = long.metric.map({'imports_gwh':'Imports','load_shedding_gwh':'Unserved demand'})
fig = px.bar(long, x='case_label', y='gwh', color='metric', barmode='group',
             labels={'case_label':'2030 demand case','gwh':'Annual energy (GWh)','metric':'Outcome'},
             title='Higher demand turns a manageable system into material unmet demand',
             color_discrete_map={'Imports':'#9467bd','Unserved demand':'#d62728'},
             hover_data={'gwh':':.1f'})
fig.update_layout(template='plotly_white')
fig.show()

**Insight:** with the same 3,666.55 MW renewable portfolio and the same modeled topology, Expansion produces 1,894.79 GWh of shedding versus 124.99 GWh in Tendencial. The exercise isolates demand growth as the main differentiator—not a change in renewable build-out.

## 5. Duration curves: how many hours are truly severe?

Duration curves rank hourly stress from highest to lowest. They distinguish a short, sharp peak from a persistent operating problem.

In [9]:
fig = px.line(duration, x='rank_hour', y='combined_mw_desc', color='scenario',
              labels={'rank_hour':'Ranked hour (1 = highest stress)','combined_mw_desc':'Import + shedding (MW)','scenario':'Case'},
              title='Stress duration curves show both severity and persistence',
              hover_data={'imports_mw_desc':':.1f','shedding_mw_desc':':.1f'})
fig.update_xaxes(range=[1,1500])
fig.update_layout(template='plotly_white')
fig.show()

**Insight:** the worst hours are coincident import-cap and shedding hours. This is a useful operational signal: additional annual energy is not enough if it cannot be delivered during the same peak-stress hours.

## 6. Model diagnostic: where does the 2030 model assign shedding?

This map is deliberately labeled as a model diagnostic. It is useful for inspecting the result, but it is not yet evidence of real-world causal geography because the scenario uses proxy placement and the 2022 topology.

In [10]:
nodal_frames = []
for case in ['tendencial','expansion']:
    d = load(f'{case}_2030_nodal_shedding.csv').assign(case=case.title())
    nodal_frames.append(d)
nodal = pd.concat(nodal_frames, ignore_index=True)
nodal = nodal[nodal.shed_gwh > 0]
fig = px.scatter_map(nodal, lat='y', lon='x', size='shed_gwh', color='case',
                     hover_name='bus', hover_data={'shed_gwh':':.2f','x':False,'y':False},
                     zoom=5, center={'lat':-1.5,'lon':-78.5},
                     map_style='open-street-map',
                     labels={'shed_gwh':'Shedding (GWh)','case':'2030 case'},
                     title='Nodal shedding assignment: inspect, do not over-interpret')
fig.update_layout(template='plotly_white', height=675)
fig.show()

**Insight:** the map identifies where the current mathematical model places unmet demand. The next analytical step would be to test whether those locations remain stable after reviewed project connections, transmission reinforcements, and demand geography are introduced.

## 7. Animated hydro-scenario maps

The saved H00–H50 networks contain hourly load-shedding dispatch. These maps use a Plotly slider: choose a hydro scenario and hover over each bus.

In [11]:
import pypsa
hydro_dir = root / 'results' / 'phase_08a'
scenario_order = list(annual.scenario)
peak_rows, total_rows = [], []
for scenario in scenario_order:
    network = pypsa.Network(str(hydro_dir / f'{scenario}.solved.nc'))
    shed_ids = network.generators.index[network.generators.carrier.eq('load_shedding')]
    dispatch = network.generators_t.p[shed_ids].clip(lower=0)
    by_bus = dispatch.groupby(network.generators.loc[shed_ids, 'bus'], axis=1).sum()
    for bus in by_bus.columns:
        peak_rows.append({'scenario':scenario,'bus':str(bus),'x':network.buses.at[bus,'x'],'y':network.buses.at[bus,'y'],'value':by_bus[bus].max()})
        total_rows.append({'scenario':scenario,'bus':str(bus),'x':network.buses.at[bus,'x'],'y':network.buses.at[bus,'y'],'value':by_bus[bus].sum()/1000})
peak_map = pd.DataFrame(peak_rows).query('value > 0')
total_map = pd.DataFrame(total_rows).query('value > 0')

INFO:pypsa.io:Imported network H00.solved.nc has buses, carriers, generators, lines, links, loads, transformers
C:\Users\fpenaherrera_vaca\AppData\Local\Temp\ipykernel_26392\392697043.py:9: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.

INFO:pypsa.io:Imported network H10.solved.nc has buses, carriers, generators, lines, links, loads, transformers
C:\Users\fpenaherrera_vaca\AppData\Local\Temp\ipykernel_26392\392697043.py:9: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.

INFO:pypsa.io:Imported network H20.solved.nc has buses, carriers, generators, lines, links, loads, transformers
C:\Users\fpenaherrera_vaca\AppData\Local\Temp\ipykernel_26392\392697043.py:9: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.

INFO:pypsa.io:Imported network H30.solved.nc has buses, carriers, generators, lines, links, loads, t

In [12]:
fig = px.scatter_map(peak_map, lat='y', lon='x', size='value', color='value', animation_frame='scenario', category_orders={'scenario':scenario_order}, hover_name='bus', hover_data={'value':':.1f'}, color_continuous_scale='Reds', zoom=5, center={'lat':-1.5,'lon':-78.5}, map_style='open-street-map', labels={'value':'Peak unserved demand (MW)'}, title='Peak unserved demand by bus — use the scenario slider')
fig.update_layout(template='plotly_white', height=675)
fig.show()

**Insight:** increasing hydro stress changes both the intensity and spatial pattern of the worst hourly deficit. This is the map for communicating acute operational exposure.

In [13]:
fig = px.scatter_map(total_map, lat='y', lon='x', size='value', color='value', animation_frame='scenario', category_orders={'scenario':scenario_order}, hover_name='bus', hover_data={'value':':.2f'}, color_continuous_scale='OrRd', zoom=5, center={'lat':-1.5,'lon':-78.5}, map_style='open-street-map', labels={'value':'Total unserved demand (GWh)'}, title='Total unserved demand by bus — use the scenario slider')
fig.update_layout(template='plotly_white', height=675)
fig.show()

**Insight:** peak MW and total GWh answer different questions: peak MW identifies locations needing instantaneous support; annual GWh identifies persistent service shortfalls.

## 8. Peak transmission loading across the full simulation

This map uses the maximum hourly absolute line flow across the complete 2022 simulation, divided by the modeled static line limit. The color scale is capped at 1.0 per unit so overloaded lines remain visibly saturated.

In [14]:
network = pypsa.Network(str(root / 'results' / 'phase_06' / 'network_solved_2022.nc'))
line_flow = network.lines_t.p0.abs()
loading = line_flow.div(network.lines.s_nom.replace(0, pd.NA), axis=1)
peak_loading = loading.max()
line_rows = []
for line, peak in peak_loading.items():
    row = network.lines.loc[line]; b0, b1 = row.bus0, row.bus1
    line_rows.append({'line':line,'x0':network.buses.at[b0,'x'],'y0':network.buses.at[b0,'y'],'x1':network.buses.at[b1,'x'],'y1':network.buses.at[b1,'y'],'peak_loading_pu':peak,'peak_loading_pct':peak*100})
line_map = pd.DataFrame(line_rows).sort_values('peak_loading_pu', ascending=False)
line_map.head(10)[['line','peak_loading_pu','peak_loading_pct']]

INFO:pypsa.io:Imported network network_solved_2022.nc has buses, carriers, generators, lines, links, loads, transformers


,line,peak_loading_pu,peak_loading_pct
111,repair_line_186_86_la_concordia_santo_domingo,1.0,100.0
110,repair_line_62_186_quininde_la_concordia,1.0,100.0
57,100,0.7,70.0
11,14,0.7,70.0
33,47,0.7,70.0
46,69,0.7,70.0
50,84,0.7,70.0
54,93,0.7,70.0
28,36,0.7,70.0
74,137,0.7,70.0


In [15]:
fig = go.Figure()
for _, row in line_map.iterrows():
    intensity = max(0, min(255, int(255 * min(row.peak_loading_pu, 1) )))
    color = f'rgb({210 - int(210 * intensity / 255)},{235 - int(200 * intensity / 255)},{255 - int(135 * intensity / 255)})'
    fig.add_trace(go.Scattermap(lon=[row.x0,row.x1], lat=[row.y0,row.y1], mode='lines', line={'width':2,'color':color}, hovertemplate=f"{row.line}<br>Peak loading: {row.peak_loading_pu:.2f} p.u.<br>Peak loading: {row.peak_loading_pct:.1f}%<extra></extra>"))
fig.update_layout(map={'style':'open-street-map','center':{'lat':-1.5,'lon':-78.5},'zoom':5}, title='2022 peak line loading — fixed-width lines, color capped at 1.0 p.u.', template='plotly_white', showlegend=False, height=675)
fig.show()

**Insight:** the color identifies the most stressed line in the full simulation span. Lines at or above 1.0 p.u. are overloaded in the modeled dispatch; because the scale is capped, all higher values remain visibly saturated. This is a linear-flow diagnostic, not an AC security or N−1 assessment.

## 9. 2022 baseline versus CENACE

Before interpreting future stress, compare the modeled 2022 baseline with the CENACE reference in both absolute energy and contribution to the total supply mix.

In [16]:
baseline = load('baseline_cenace_comparison.csv')
baseline_long = baseline.melt(id_vars='carrier', value_vars=['reference_gwh','simulated_gwh'], var_name='source', value_name='gwh')
baseline_long['source'] = baseline_long.source.map({'reference_gwh':'CENACE','simulated_gwh':'Model'})
fig = px.bar(baseline_long, x='carrier', y='gwh', color='source', barmode='group', labels={'carrier':'Carrier','gwh':'Energy (GWh)','source':'Reference'}, title='2022 baseline: model versus CENACE energy by carrier', template='plotly_white')
fig.show()
share_cols = [c for c in baseline.columns if 'share' in c.lower()]
print('Available contribution/share columns:', share_cols)
baseline[['carrier','reference_gwh','simulated_gwh','signed_error_gwh','relative_error','signed_error_pp']]

Available contribution/share columns: ['reference_share', 'simulated_share']


,carrier,reference_gwh,simulated_gwh,signed_error_gwh,relative_error,signed_error_pp
0,hydro,24499.54,23919.364646,-5.801754e+02,-2.368107e-02,0.775950
1,thermal,3393.35,3229.487738,-1.638623e+02,-4.828923e-02,-0.193407
2,nonconventional,325.89,138.507616,-1.873824e+02,-5.749866e-01,-0.637045
3,interconnection,465.83,465.830000,-5.684342e-14,-1.220261e-16,0.054502


In [17]:
share_view = baseline.melt(id_vars='carrier', value_vars=['reference_share','simulated_share'], var_name='source', value_name='share_pct')
share_view['share_pct'] = share_view.share_pct * 100
share_view['source'] = share_view.source.map({'reference_share':'CENACE contribution','simulated_share':'Model contribution'})
fig = px.bar(share_view, x='carrier', y='share_pct', color='source', barmode='group', labels={'carrier':'Carrier','share_pct':'Contribution to total supply (%)','source':'Reference'}, title='2022 baseline: contribution to total supply mix', template='plotly_white')
fig.show()

**Insight:** the GWh comparison shows absolute scale, while the contribution comparison shows composition. A category can be close in energy but still have a meaningful share difference—or vice versa. The nonconventional category is the clearest baseline validation gap and should remain visible when interpreting future scenarios.

## Takeaway

The results support a clear demonstrative analytics story: the system first substitutes between resources, then reaches constraints, after which stress grows rapidly. The most valuable next experiment is a two-dimensional matrix combining demand growth with hydro reduction, followed by the same interactive KPI, map, and duration-curve views.